In [1]:
%run initialize_environment.py

MCP Windows stderr patch applied.
Azure OpenAI LLM module imported successfully.
Environment variables loaded successfully. Initializing completed successfully.


## Local MCP server

In [2]:
client =  MultiServerMCPClient(
        {
            "local_server": {
                    "transport": "stdio",
                    "command": "python",
                    "args": ["resources/mcp_server_base.py"],
                }
        }
    )

server = "local_server"

# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources(server)

# get prompts
prompt = await client.get_prompt(server, "prompt")

In [3]:
llm = create_azure_llm()

agent = create_agent(
    llm,
    tools=tools,
    system_prompt=prompt[0].content
)

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

response

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='b789376b-f443-4a93-aae9-6dc3d573937c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 188, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DNayd43WPG9jtbJ6HbwIbXsva7hF0', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019d2966-29ef-7723-b16e-6c72e9c0d26b-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'call_SiyeaBl5RfEa7ioWDwvc6zV8', 'type': 'tool_call'}], invalid_tool_cal

## Connect to an online external/3rd Party MCP Servers

MCP Time Server: [https://mcpservers.org/servers/modelcontextprotocol/time](https://mcp.so/server/time/modelcontextprotocol)

### What is `uvx`?

`uvx` is a command-line tool from the [uv](https://github.com/astral-sh/uv) project that allows you to run Python packages directly without installing them globally or in a virtual environment. It's similar to `npx` in Node.js. When you run `uvx package-name`, it:

1. Downloads the package if not already cached
2. Creates an isolated environment
3. Executes the package's command-line interface
4. Cleans up after execution

This is particularly useful for running standalone tools and MCP servers without affecting your existing Python environment.
In this notebook, the MCP client starts the server with:

- command: `uvx`
- args: `mcp-server-time --local-timezone=America/New_York`

This means `uvx` resolves and runs the `mcp-server-time` package executable, and the client communicates with it over standard input/output as an MCP transport.

In [8]:
time_client = MultiServerMCPClient(
    {
        # Below JSON is available directly from the mcp server link above:
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

time_tools = await time_client.get_tools()

agent = create_agent(
    llm,
    tools=time_tools,
)

question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='d4e2674d-8b53-4b1a-b121-68b2321c43fc'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 215, 'total_tokens': 234, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DNb8KiY2uRJBuDAAWzm4upVtUPpWG', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019d296f-55d6-7413-91bc-02c00255436f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_7Q7Lx5TKzwOUMJjeG3kKn8cw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'i